# 21CMA 五个 Starlink VHF 下行频点搜寻

这个 Notebook 面向当前五组 21CMA 数据的可视化、可中断搜寻。正式算法保存在同目录的 `search_starlink_vhf_downlinks.py` 中；Notebook 逐步调用其中函数。

完整流程：

1. 清点原始文件并建立各数据集连续时间轴；
2. 用短时 FFT 同时扫描五个协调的 Starlink VHF 下行中心频率；
3. 用三种互补规则形成紧凑候选；
4. 用全局过载时间掩膜排除无法可靠评估的时段；
5. 可选地用粗峰值频率轨迹建立固定窄带线审计表，但不以此删除正式搜索候选；
6. 对所有非过载候选使用原始电压的20 ms短窗STFT，搜索预期斜率的LoRa chirp；
7. 只对斜线形态阳性候选做短窗 DDC，检查 LoRa 前导、SFD 和 Starlink `0x12` sync word；
8. 只对sync预验证阳性候选调用 LFdemod 完成 PHY header、payload、CRC 和 Starlink profile 验证；
9. 对相邻频道和重复候选解出的同一物理 packet 去重，并保存分层审计结果。

> **Data requirement.** This notebook reproduces the paper's full five-window search and requires about 2.4 TB of raw voltages. The two public Zenodo event files alone cannot reproduce the candidate counts.

## 1. 导入正式搜寻模块


In [ ]:
from pathlib import Path
import os
import sys
from datetime import datetime, timezone
import importlib.util
import json
import math
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

%matplotlib inline

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc
SEARCH_DIR = PROJECT_ROOT / '2_signal_analysis'
SCRIPT_PATH = SEARCH_DIR / 'search_starlink_vhf_downlinks.py'

spec = importlib.util.spec_from_file_location('starlink_five_channel_search', SCRIPT_PATH)
search = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = search
spec.loader.exec_module(search)

print('Loaded:', SCRIPT_PATH)

## 2. 设置本次运行目录和参数

每次正式重跑建议更换 `RUN_NAME`，不要覆盖旧结果。默认处理五组观测和五个协调的 space-to-Earth 中心频率。

默认搜索参数：

- 原始采样率：480 MS/s，实数 `int16`；
- FFT：65,536 点 Hann 窗；
- 相邻快照间隔：25 ms；
- 可选稀疏形态审计：至少 4 个粗峰值 bin，频跨至少 25 kHz；该层只用于加速评估，不改变正式chirp搜索输入；
- 原始 chirp 形态：每个候选最多 4 个 20 ms 窗，搜索约 +6.78 kHz/ms 的短斜线；
- 前导预验证窗口：不少于 1.6 s；
- 完整 LFdemod 验证窗口：5.5 s；
- 不默认保存 240 kS/s 中间 IQ。

In [ ]:
DATA_ROOT = Path(os.environ.get('FULL_SURVEY_DATA_ROOT', PROJECT_ROOT / 'data' / 'full_survey'))
DATASETS = list(search.DEFAULT_DATASETS)
RUN_NAME = 'five_downlinks_notebook_v1'
OUTPUT_DIR = RESULTS_DIR / '2_full_survey_search' / RUN_NAME
RETAIN_IQ = False

config = search.SearchConfig(
    nfft=65_536,
    step_seconds=0.025,
    verification_window_seconds=5.5,
)
config.validate()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

display(pd.DataFrame({
    'channel_MHz': np.asarray(config.channels_hz) / 1e6,
}))
print('Output directory:', OUTPUT_DIR)
print('FFT duration (ms):', config.nfft / config.sample_rate_hz * 1e3)
print('FFT-bin spacing (kHz):', config.sample_rate_hz / config.nfft / 1e3)

## 3. 清点原始数据并建立时间轴

In [ ]:
inventory = search.build_inventory(DATA_ROOT, DATASETS, config)
inventory_df = pd.DataFrame(inventory)

search._write_csv(
    OUTPUT_DIR / 'dataset_inventory.csv',
    inventory,
    search.INVENTORY_FIELDS,
)
search._write_json(
    OUTPUT_DIR / 'search_config.json',
    {
        'created_utc': datetime.now(timezone.utc).isoformat(),
        'data_root': str(DATA_ROOT.resolve()),
        'datasets': DATASETS,
        'search_config': search.asdict(config),
        'interface': 'Jupyter Notebook',
    },
)

inventory_summary = (
    inventory_df.groupby('dataset', as_index=False)
    .agg(
        file_count=('file_number', 'count'),
        first_file=('file_number', 'min'),
        last_file=('file_number', 'max'),
        duration_s=('duration_s', 'sum'),
        continuous_segments=('segment_id', lambda values: values.max() + 1),
    )
)
display(inventory_summary)
print('Total files:', len(inventory_df))
print('Total duration (min):', inventory_df['duration_s'].sum() / 60.0)

## 4. 建立频谱搜索掩膜

每个 65,536 点 FFT 只覆盖约 0.1365 ms。同一张 FFT 同时检查五个频道。目标频带取中心 ±35 kHz；参考背景取 136.98–138.04 MHz，并排除五个频道中心附近 ±50 kHz。

每个目标频道使用最强三个 FFT bin 的平均功率，与参考频带功率中位数比较。之后再减去该文件、该频道随时间统计量的中位数，得到用于候选筛选的瞬态 `excess_db`。

In [ ]:
frequencies, target_indices, reference_mask = search._search_masks(config)

mask_rows = []
for channel_hz, indices in zip(config.channels_hz, target_indices):
    mask_rows.append({
        'channel_MHz': channel_hz / 1e6,
        'target_bin_count': len(indices),
        'first_target_bin_MHz': frequencies[indices[0]] / 1e6,
        'last_target_bin_MHz': frequencies[indices[-1]] / 1e6,
    })
display(pd.DataFrame(mask_rows))
print('Reference-bin count:', int(reference_mask.sum()))

## 5. 初始化可中断扫描状态

`hits_by_file` 以文件路径为键保存已经完成的结果。第一次运行会创建空字典；扫描过程中按文件写入内存。若在下一格按 Kernel Interrupt，中断后重新运行扫描格，会跳过本内核会话中已经完成的文件。

如果确实希望从头重新扫描，请手动执行注释中的 `hits_by_file = {}`。

In [ ]:
if 'hits_by_file' not in globals():
    hits_by_file = {}

# 若要在当前内核中完全重置扫描，取消下一行注释并单独运行一次。
# hits_by_file = {}

print(f'Completed files already in memory: {len(hits_by_file)}/{len(inventory)}')

## 6. 执行或继续五频点稀疏扫描

每个文件只读一次，每一个短 FFT 同时计算五个频点。这里只保留 `excess_db ≥ 4 dB` 的稀疏命中，不向磁盘写全量快照表。

中断方法：在 Jupyter 中点击停止按钮或执行 **Kernel → Interrupt**。中断后不要重新运行上一格的重置语句，直接重新运行本格即可继续。

In [ ]:
for file_index, record in enumerate(inventory, start=1):
    key = str(record['input_path'])
    if key in hits_by_file:
        continue
    print(
        f"[{file_index}/{len(inventory)}] "
        f"{record['dataset']}/{record['file_name']}",
        flush=True,
    )
    hits_by_file[key] = search.scan_one_file(
        record,
        config,
        frequencies,
        target_indices,
        reference_mask,
    )

print(f'Completed files: {len(hits_by_file)}/{len(inventory)}')
print('Retained sparse hits:', sum(len(rows) for rows in hits_by_file.values()))

## 7. 从稀疏命中形成候选

为了避免只用一个 8 dB 阈值漏掉第二个已知弱事件，满足以下任一规则就进入后续数据质量、形态与解码验证链：

1. `strong_single`：至少一个快照 ≥ 8 dB；
2. `persistent_weak`：1.2 s 内至少三个快照 ≥ 4 dB；
3. `medium_pair`：0.8 s 内至少两个快照 ≥ 6 dB。

这一格要求 293 个文件全部扫描完成，以防误把部分数据结果当成全数据结论。两个已知 137.055 MHz 包只作为回归检查，不参与阈值拟合。

In [ ]:
if len(hits_by_file) != len(inventory):
    raise RuntimeError(
        f'Only {len(hits_by_file)}/{len(inventory)} files have been scanned. '
        'Finish Section 6 before forming the formal candidate table.'
    )

all_hits = [row for rows in hits_by_file.values() for row in rows]
candidates = search.form_candidates(all_hits, config)
energy_candidates = [dict(row) for row in candidates]
candidate_df = pd.DataFrame(candidates)
known_checks = search._known_search_recovery(candidates)

search._write_csv(
    OUTPUT_DIR / 'candidates.csv',
    candidates,
    search.CANDIDATE_FIELDS,
)
search.save_candidate_overview(
    OUTPUT_DIR / 'candidate_overview.png',
    candidates,
    DATASETS,
)
search._write_json(
    OUTPUT_DIR / 'scan_summary.json',
    {
        'file_count': len(inventory),
        'total_duration_s': float(inventory_df['duration_s'].sum()),
        'retained_above_floor_hit_count_in_memory': len(all_hits),
        'candidate_count': len(candidates),
        'known_search_checks': known_checks,
        'dense_sparse_samples_saved': False,
        'scope_limit': 'No injection/recovery completeness test was performed.',
    },
)

display(pd.DataFrame(known_checks))
display(candidate_df.head(30))
print('Candidate count:', len(candidate_df))
display(Image(filename=str(OUTPUT_DIR / 'candidate_overview.png')))

## 8. 按数据集和频点检查候选数量

在开始计算量更高的过载、chirp形态和 LFdemod 验证之前，先检查候选是否集中在某个数据集或频点，以及候选数量是否异常庞大。能量候选不是 Starlink 结论。

In [ ]:
if candidate_df.empty:
    print('No energy candidates passed the configured rules.')
else:
    candidate_count_table = pd.pivot_table(
        candidate_df.assign(
            channel_MHz=candidate_df['rf_frequency_hz'] / 1e6,
            count=1,
        ),
        index='dataset',
        columns='channel_MHz',
        values='count',
        aggfunc='sum',
        fill_value=0,
    )
    display(candidate_count_table)
    display(
        candidate_df.sort_values('peak_excess_db', ascending=False)
        .head(20)[[
            'candidate_id', 'dataset', 'rf_frequency_hz',
            'peak_file_number', 'peak_file_time_s',
            'peak_excess_db', 'qualification_rules',
        ]]
    )

## 9. 建立过载时间掩膜并过滤候选

这一节是针对本批21CMA数据存在强宽带过载、模拟前端压缩和互调条纹而增加的数据质量控制。它不依赖Starlink候选，先在全部原始数据上独立确定哪些时间不能可靠分析，再用这些连续时间区间过滤第7节候选。

处理原则：

1. 每25 ms读取一个65,536点短块，计算ADC轨道命中、接近满量程比例、p99.9、RMS、75–200 MHz总功率，并将该频段分成125个1 MHz子带；
2. 每个数据集用所有文件的稀疏样本建立稳健中位数背景，避免把单个长过载文件自身当作“正常基线”；
3. 宽带判据要求75–200 MHz总功率至少高于背景2 dB，且至少6个独立1 MHz子带同时高于各自背景2 dB；这可以抑制把单个真实强窄带信号直接当成过载。ADC轨道判据要求满量程样本比例至少为 `1e-4`，避免仅凭一两个孤立轨道命中删掉可解码的真事件；
4. 将相邻异常采样点合并为连续过载区间，并在区间两端各增加0.1 s保护时间；
5. 候选的能量命中时间范围只要与过载区间相交，就进入 `overload_masked_candidates.csv`，不再进入chirp形态和LFdemod验证；
6. 过滤只表示“该时段无法可靠评估”，不能把它写成“没有Starlink发射”。



In [ ]:
# 9.1 配置过载判据并建立每个数据集的稳健背景
OVERLOAD_SCAN_STEP_SECONDS = 0.025
OVERLOAD_NFFT = config.nfft
OVERLOAD_BAND_HZ = (75e6, 200e6)
OVERLOAD_BIN_EXCESS_DB = 6.0
OVERLOAD_BROADBAND_EXCESS_DB = 2.0
OVERLOAD_SUBBAND_WIDTH_HZ = 1e6
OVERLOAD_SUBBAND_EXCESS_DB = 2.0
OVERLOAD_MIN_ELEVATED_SUBBANDS = 6
OVERLOAD_NEAR_FULL_SCALE_FRACTION = 1e-4
OVERLOAD_CLIP_FRACTION = 1e-4
OVERLOAD_P99P9_FULL_SCALE = 0.60
OVERLOAD_RMS_FULL_SCALE = 0.20
OVERLOAD_BASELINE_BLOCKS_PER_FILE = 2
OVERLOAD_MERGE_GAP_SECONDS = 0.20
OVERLOAD_EDGE_GUARD_SECONDS = 0.10

_overload_half = OVERLOAD_NFFT // 2
_overload_window = np.hanning(OVERLOAD_NFFT).astype(np.float32)
_overload_frequencies = np.fft.rfftfreq(
    OVERLOAD_NFFT, d=1.0 / config.sample_rate_hz
)
_overload_band_mask = (
    (_overload_frequencies >= OVERLOAD_BAND_HZ[0])
    & (_overload_frequencies <= OVERLOAD_BAND_HZ[1])
)
_overload_subband_edges_hz = np.arange(
    OVERLOAD_BAND_HZ[0],
    OVERLOAD_BAND_HZ[1] + OVERLOAD_SUBBAND_WIDTH_HZ,
    OVERLOAD_SUBBAND_WIDTH_HZ,
)
_overload_subband_slices = [
    slice(
        int(np.searchsorted(_overload_frequencies, low_hz, side='left')),
        int(np.searchsorted(_overload_frequencies, high_hz, side='left')),
    )
    for low_hz, high_hz in zip(
        _overload_subband_edges_hz[:-1],
        _overload_subband_edges_hz[1:],
    )
]


def _overload_blocks(record, center_times_s):
    """Read bounded int16 blocks from one raw file through a read-only memmap."""
    raw = np.memmap(record['input_path'], dtype=config.dtype, mode='r')
    blocks = []
    earliest = _overload_half / config.sample_rate_hz
    latest = (len(raw) - _overload_half) / config.sample_rate_hz
    for center_s in center_times_s:
        bounded = float(np.clip(center_s, earliest, latest))
        center_sample = int(round(bounded * config.sample_rate_hz))
        blocks.append(np.asarray(
            raw[center_sample - _overload_half:center_sample + _overload_half],
            dtype=np.int16,
        ).copy())
    del raw
    return np.stack(blocks)


def _overload_power(blocks):
    voltage = blocks.astype(np.float32)
    spectrum = np.fft.rfft(voltage * _overload_window[None, :], axis=1)
    return (np.abs(spectrum) ** 2).astype(np.float32)


def _overload_subband_powers(powers):
    """Integrate a power spectrum into independent 1 MHz subbands."""
    return np.stack([
        powers[:, subband_slice].sum(axis=1, dtype=np.float64)
        for subband_slice in _overload_subband_slices
    ], axis=1)


def _build_dataset_overload_baseline(dataset):
    """Return a dataset-wide median spectrum, band power and RMS baseline."""
    sampled_powers = []
    sampled_rms = []
    records = [row for row in inventory if row['dataset'] == dataset]
    for record in records:
        edge = _overload_half / config.sample_rate_hz
        centers = np.linspace(
            edge,
            float(record['duration_s']) - edge,
            OVERLOAD_BASELINE_BLOCKS_PER_FILE + 2,
        )[1:-1]
        blocks = _overload_blocks(record, centers)
        sampled_powers.append(_overload_power(blocks))
        sampled_rms.extend(
            np.sqrt(np.mean(blocks.astype(np.float64) ** 2, axis=1)).tolist()
        )
    powers = np.concatenate(sampled_powers, axis=0)
    band_powers = powers[:, _overload_band_mask].sum(axis=1, dtype=np.float64)
    subband_powers = _overload_subband_powers(powers)
    return {
        'median_power_per_bin': np.median(powers, axis=0),
        'median_75_200MHz_power': float(np.median(band_powers)),
        'median_1MHz_subband_power': np.median(subband_powers, axis=0),
        'median_rms': float(np.median(sampled_rms)),
        'baseline_block_count': int(len(powers)),
        'file_count': int(len(records)),
    }


overload_baselines = {}
for dataset in DATASETS:
    print('Building overload baseline:', dataset, flush=True)
    overload_baselines[dataset] = _build_dataset_overload_baseline(dataset)

baseline_summary = {
    dataset: {
        key: value for key, value in baseline.items()
        if key not in {'median_power_per_bin', 'median_1MHz_subband_power'}
    }
    for dataset, baseline in overload_baselines.items()
}
search._write_json(
    OUTPUT_DIR / 'overload_baseline_summary.json',
    baseline_summary,
)
display(pd.DataFrame.from_dict(baseline_summary, orient='index'))
print('Saved:', OUTPUT_DIR / 'overload_baseline_summary.json')

In [ ]:
# 9.2 可中断地扫描全部文件，识别过载采样点
# 同一内核中断后重新运行本格，会跳过已经完成的文件。
# 若第9.1节的判据参数改变，参数签名会自动使旧缓存失效。
current_overload_scan_signature = (
    OVERLOAD_SCAN_STEP_SECONDS, OVERLOAD_NFFT, OVERLOAD_BAND_HZ,
    OVERLOAD_BIN_EXCESS_DB, OVERLOAD_BROADBAND_EXCESS_DB,
    OVERLOAD_SUBBAND_WIDTH_HZ, OVERLOAD_SUBBAND_EXCESS_DB,
    OVERLOAD_MIN_ELEVATED_SUBBANDS,
    OVERLOAD_NEAR_FULL_SCALE_FRACTION, OVERLOAD_CLIP_FRACTION,
    OVERLOAD_P99P9_FULL_SCALE, OVERLOAD_RMS_FULL_SCALE,
)
if globals().get('overload_scan_signature') != current_overload_scan_signature:
    overload_samples_by_file = {}
    overload_scan_signature = current_overload_scan_signature
    print('Overload criteria changed: reset the per-file scan cache.')
elif 'overload_samples_by_file' not in globals():
    overload_samples_by_file = {}


def _scan_file_for_overload(record):
    """Return only flagged overload samples from one file."""
    raw_sample_count = int(record['sample_count'])
    first_time = _overload_half / config.sample_rate_hz
    last_time = (raw_sample_count - _overload_half) / config.sample_rate_hz
    times = np.arange(
        first_time, last_time, OVERLOAD_SCAN_STEP_SECONDS, dtype=np.float64
    )
    baseline = overload_baselines[record['dataset']]
    tiny = np.finfo(np.float64).tiny
    flagged_rows = []
    batch_size = 32
    for batch_start in range(0, len(times), batch_size):
        batch_times = times[batch_start:batch_start + batch_size]
        blocks = _overload_blocks(record, batch_times)
        samples = blocks.astype(np.int32)
        voltage = blocks.astype(np.float64)
        powers = _overload_power(blocks)

        clip_fraction = np.mean(
            (samples == 32767) | (samples == -32768), axis=1
        )
        near_full_fraction = np.mean(
            np.abs(samples) >= 0.99 * 32768.0, axis=1
        )
        p99p9 = np.percentile(np.abs(samples), 99.9, axis=1) / 32768.0
        rms = np.sqrt(np.mean(voltage ** 2, axis=1)) / 32768.0
        rms_excess_db = 20.0 * np.log10(
            np.maximum(rms * 32768.0, tiny)
            / max(baseline['median_rms'], tiny)
        )
        band_power = powers[:, _overload_band_mask].sum(axis=1, dtype=np.float64)
        broadband_excess_db = 10.0 * np.log10(
            np.maximum(band_power, tiny)
            / max(baseline['median_75_200MHz_power'], tiny)
        )
        spectral_excess_db = 10.0 * np.log10(
            np.maximum(powers[:, _overload_band_mask], tiny)
            / np.maximum(
                baseline['median_power_per_bin'][None, _overload_band_mask], tiny
            )
        )
        occupancy = np.mean(
            spectral_excess_db >= OVERLOAD_BIN_EXCESS_DB, axis=1
        )
        subband_power = _overload_subband_powers(powers)
        subband_excess_db = 10.0 * np.log10(
            np.maximum(subband_power, tiny)
            / np.maximum(
                baseline['median_1MHz_subband_power'][None, :], tiny
            )
        )
        elevated_subband_count = np.sum(
            subband_excess_db >= OVERLOAD_SUBBAND_EXCESS_DB, axis=1
        )
        maximum_subband_excess_db = np.max(subband_excess_db, axis=1)

        rail = clip_fraction >= OVERLOAD_CLIP_FRACTION
        near_full = near_full_fraction >= OVERLOAD_NEAR_FULL_SCALE_FRACTION
        analog_compression = (
            (p99p9 >= OVERLOAD_P99P9_FULL_SCALE)
            & (rms >= OVERLOAD_RMS_FULL_SCALE)
        )
        broadband_overload = (
            (broadband_excess_db >= OVERLOAD_BROADBAND_EXCESS_DB)
            & (elevated_subband_count >= OVERLOAD_MIN_ELEVATED_SUBBANDS)
        )
        flagged = rail | near_full | analog_compression | broadband_overload

        for local_index in np.flatnonzero(flagged):
            reasons = []
            if rail[local_index]:
                reasons.append('adc_rail_hits')
            if near_full[local_index]:
                reasons.append('near_full_scale')
            if analog_compression[local_index]:
                reasons.append('analog_compression')
            if broadband_overload[local_index]:
                reasons.append('broadband_power_and_multiple_subbands')
            file_time_s = float(batch_times[local_index])
            flagged_rows.append({
                'dataset': record['dataset'],
                'segment_id': int(record['segment_id']),
                'file_number': int(record['file_number']),
                'file_time_s': file_time_s,
                'dataset_time_s': float(record['dataset_start_s']) + file_time_s,
                'clip_fraction': float(clip_fraction[local_index]),
                'near_full_scale_fraction': float(near_full_fraction[local_index]),
                'p99p9_abs_fraction_full_scale': float(p99p9[local_index]),
                'rms_fraction_full_scale': float(rms[local_index]),
                'rms_excess_db': float(rms_excess_db[local_index]),
                'power_75_200MHz_excess_db': float(broadband_excess_db[local_index]),
                'occupancy_75_200MHz_fraction': float(occupancy[local_index]),
                'elevated_1MHz_subband_count': int(
                    elevated_subband_count[local_index]
                ),
                'maximum_1MHz_subband_excess_db': float(
                    maximum_subband_excess_db[local_index]
                ),
                'overload_reasons': ';'.join(reasons),
            })
    return flagged_rows


for file_index, record in enumerate(inventory, start=1):
    key = str(record['input_path'])
    if key in overload_samples_by_file:
        continue
    print(
        f"[overload scan {file_index}/{len(inventory)}] "
        f"{record['dataset']}/{record['file_name']}",
        flush=True,
    )
    overload_samples_by_file[key] = _scan_file_for_overload(record)

print(
    f"Completed files: {len(overload_samples_by_file)}/{len(inventory)}"
)
print(
    'Flagged overload samples:',
    sum(len(rows) for rows in overload_samples_by_file.values()),
)

In [ ]:
# 9.3 合并连续过载区间、过滤候选并保存审计结果
if len(overload_samples_by_file) != len(inventory):
    raise RuntimeError(
        f'Only {len(overload_samples_by_file)}/{len(inventory)} files have '
        'overload results. Finish Section 9.2 first.'
    )

overload_flagged_samples = [
    row for rows in overload_samples_by_file.values() for row in rows
]
overload_flagged_samples.sort(key=lambda row: (
    row['dataset'], int(row['segment_id']), float(row['dataset_time_s'])
))

# Group adjacent flagged samples on the continuous per-dataset timeline.
groups = []
for row in overload_flagged_samples:
    if (
        not groups
        or row['dataset'] != groups[-1][-1]['dataset']
        or int(row['segment_id']) != int(groups[-1][-1]['segment_id'])
        or float(row['dataset_time_s']) - float(groups[-1][-1]['dataset_time_s'])
           > OVERLOAD_MERGE_GAP_SECONDS
    ):
        groups.append([row])
    else:
        groups[-1].append(row)


def _segment_bounds(dataset, segment_id):
    records = [
        row for row in inventory
        if row['dataset'] == dataset and int(row['segment_id']) == int(segment_id)
    ]
    return (
        min(float(row['dataset_start_s']) for row in records),
        max(float(row['dataset_stop_s']) for row in records),
    )


def _locate_dataset_time(dataset, segment_id, time_s):
    records = [
        row for row in inventory
        if row['dataset'] == dataset and int(row['segment_id']) == int(segment_id)
    ]
    for record in records:
        if (
            float(record['dataset_start_s']) <= time_s
            <= float(record['dataset_stop_s']) + 1e-9
        ):
            local = float(np.clip(
                time_s - float(record['dataset_start_s']),
                0.0,
                float(record['duration_s']),
            ))
            return int(record['file_number']), local
    raise ValueError(f'Cannot locate dataset time {dataset} {segment_id} {time_s}')


overload_intervals = []
for interval_index, group in enumerate(groups, start=1):
    dataset = group[0]['dataset']
    segment_id = int(group[0]['segment_id'])
    segment_start, segment_stop = _segment_bounds(dataset, segment_id)
    start_s = max(
        segment_start,
        float(group[0]['dataset_time_s'])
        - OVERLOAD_SCAN_STEP_SECONDS / 2.0
        - OVERLOAD_EDGE_GUARD_SECONDS,
    )
    stop_s = min(
        segment_stop,
        float(group[-1]['dataset_time_s'])
        + OVERLOAD_SCAN_STEP_SECONDS / 2.0
        + OVERLOAD_EDGE_GUARD_SECONDS,
    )
    start_file, start_local = _locate_dataset_time(dataset, segment_id, start_s)
    stop_file, stop_local = _locate_dataset_time(dataset, segment_id, stop_s)
    reasons = sorted({
        reason
        for row in group
        for reason in str(row['overload_reasons']).split(';')
        if reason
    })
    overload_intervals.append({
        'overload_interval_id': f'O{interval_index:05d}',
        'dataset': dataset,
        'segment_id': segment_id,
        'dataset_start_s': start_s,
        'dataset_stop_s': stop_s,
        'duration_s': stop_s - start_s,
        'start_file_number': start_file,
        'start_file_time_s': start_local,
        'stop_file_number': stop_file,
        'stop_file_time_s': stop_local,
        'flagged_sample_count': len(group),
        'maximum_clip_fraction': max(float(row['clip_fraction']) for row in group),
        'maximum_near_full_scale_fraction': max(
            float(row['near_full_scale_fraction']) for row in group
        ),
        'maximum_p99p9_fraction_full_scale': max(
            float(row['p99p9_abs_fraction_full_scale']) for row in group
        ),
        'maximum_rms_fraction_full_scale': max(
            float(row['rms_fraction_full_scale']) for row in group
        ),
        'maximum_75_200MHz_power_excess_db': max(
            float(row['power_75_200MHz_excess_db']) for row in group
        ),
        'maximum_75_200MHz_occupancy_fraction': max(
            float(row['occupancy_75_200MHz_fraction']) for row in group
        ),
        'maximum_elevated_1MHz_subband_count': max(
            int(row['elevated_1MHz_subband_count']) for row in group
        ),
        'maximum_1MHz_subband_excess_db': max(
            float(row['maximum_1MHz_subband_excess_db']) for row in group
        ),
        'overload_reasons': ';'.join(reasons),
    })

# Always recover the complete Section-7 energy-candidate list when rerun.
# This avoids accidentally feeding a previously filtered `candidates` list back
# into the overload mask during interactive notebook work.
if 'energy_candidates' not in globals():
    raise RuntimeError('Rerun Section 7 before Section 9.3.')
candidates_before_overload_mask = [dict(row) for row in energy_candidates]

masked_candidates = []
retained_candidates = []
for candidate in candidates_before_overload_mask:
    # Use the stored asymmetric first/last hit times, with one scan step
    # of guard on either side.  A symmetric full-span expansion around
    # the peak can incorrectly intersect a nearby unrelated overload.
    candidate_start = (
        float(candidate['first_hit_dataset_time_s']) - config.step_seconds
    )
    candidate_stop = (
        float(candidate['last_hit_dataset_time_s']) + config.step_seconds
    )
    overlaps = [
        interval for interval in overload_intervals
        if interval['dataset'] == candidate['dataset']
        and int(interval['segment_id']) == int(candidate['segment_id'])
        and candidate_start <= float(interval['dataset_stop_s'])
        and candidate_stop >= float(interval['dataset_start_s'])
    ]
    updated = dict(candidate)
    updated['candidate_support_start_s'] = candidate_start
    updated['candidate_support_stop_s'] = candidate_stop
    updated['overload_masked'] = bool(overlaps)
    updated['overload_interval_ids'] = ';'.join(
        interval['overload_interval_id'] for interval in overlaps
    )
    updated['overload_mask_reasons'] = ';'.join(sorted({
        reason
        for interval in overlaps
        for reason in str(interval['overload_reasons']).split(';')
        if reason
    }))
    if overlaps:
        masked_candidates.append(updated)
    else:
        retained_candidates.append(updated)

candidates = sorted(
    retained_candidates,
    key=lambda row: -float(row['peak_excess_db']),
)
candidate_df = pd.DataFrame(candidates)

pd.DataFrame(overload_flagged_samples).to_csv(
    OUTPUT_DIR / 'overload_flagged_samples.csv', index=False
)
pd.DataFrame(overload_intervals).to_csv(
    OUTPUT_DIR / 'overload_intervals.csv', index=False
)
pd.DataFrame(masked_candidates).to_csv(
    OUTPUT_DIR / 'overload_masked_candidates.csv', index=False
)
candidate_df.to_csv(
    OUTPUT_DIR / 'candidates_after_overload_mask.csv', index=False
)

masked_duration_s = float(sum(
    interval['duration_s'] for interval in overload_intervals
))
total_duration_s = float(sum(float(row['duration_s']) for row in inventory))
overload_summary = {
    'total_observation_duration_s': total_duration_s,
    'overload_masked_duration_s': masked_duration_s,
    'effective_clean_duration_s': total_duration_s - masked_duration_s,
    'overload_interval_count': len(overload_intervals),
    'candidate_count_before_mask': len(candidates_before_overload_mask),
    'overload_masked_candidate_count': len(masked_candidates),
    'candidate_count_after_mask': len(candidates),
    'broadband_excess_threshold_db': OVERLOAD_BROADBAND_EXCESS_DB,
    'subband_width_hz': OVERLOAD_SUBBAND_WIDTH_HZ,
    'subband_excess_threshold_db': OVERLOAD_SUBBAND_EXCESS_DB,
    'minimum_elevated_subband_count': OVERLOAD_MIN_ELEVATED_SUBBANDS,
    'interpretation': (
        'Masked intervals are unusable for reliable Starlink assessment; '
        'they are not counted as non-detections.'
    ),
}
search._write_json(OUTPUT_DIR / 'overload_summary.json', overload_summary)

# Overview: grey = all candidates, red shading = overload, green = retained.
fig, axes = plt.subplots(
    len(DATASETS), 1,
    figsize=(15, max(3.0 * len(DATASETS), 4.0)),
    constrained_layout=True,
    squeeze=False,
)
for axis, dataset in zip(axes[:, 0], DATASETS):
    all_rows = [row for row in candidates_before_overload_mask if row['dataset'] == dataset]
    kept_rows = [row for row in candidates if row['dataset'] == dataset]
    for interval in overload_intervals:
        if interval['dataset'] == dataset:
            axis.axvspan(
                interval['dataset_start_s'], interval['dataset_stop_s'],
                color='tab:red', alpha=0.20,
            )
    if all_rows:
        axis.scatter(
            [row['peak_dataset_time_s'] for row in all_rows],
            [row['rf_frequency_hz'] / 1e6 for row in all_rows],
            s=10, color='0.65', label='All energy candidates', zorder=2,
        )
    if kept_rows:
        axis.scatter(
            [row['peak_dataset_time_s'] for row in kept_rows],
            [row['rf_frequency_hz'] / 1e6 for row in kept_rows],
            s=18, color='tab:green', label='Retained after overload mask', zorder=3,
        )
    axis.set_title(dataset)
    axis.set_ylabel('RF center (MHz)')
    handles, labels = axis.get_legend_handles_labels()
    if handles:
        axis.legend(loc='upper right', fontsize=8)
axes[-1, 0].set_xlabel('Time from start of dataset (s)')
fig.savefig(OUTPUT_DIR / 'overload_interval_overview.png', dpi=200)
plt.close(fig)

known_candidate_ids = {
    candidate_id
    for check in known_checks
    for candidate_id in check.get('matching_candidate_ids', [])
}
masked_known_ids = sorted(
    known_candidate_ids & {row['candidate_id'] for row in masked_candidates}
)

# display(Markdown('### 连续过载时间区间'))
display(pd.DataFrame(overload_intervals).head(50))
display(pd.DataFrame([overload_summary]))
if candidate_df.empty:
    retained_top30_df = candidate_df.copy()
    print('No candidates remain after the overload time mask.')
else:
    retained_top30_df = candidate_df.sort_values(
        'peak_excess_db', ascending=False
    ).head(30)
    display(Markdown('### 过载时间掩膜后保留的前30个候选'))
    display(retained_top30_df)
retained_top30_df.to_csv(
    OUTPUT_DIR / 'retained_top30_candidates.csv', index=False
)
display(Image(filename=str(OUTPUT_DIR / 'overload_interval_overview.png')))

if masked_known_ids:
    print('WARNING: known-event regression candidates were masked:', masked_known_ids)
print(json.dumps(overload_summary, ensure_ascii=False, indent=2))
print('Saved:', OUTPUT_DIR / 'overload_flagged_samples.csv')
print('Saved:', OUTPUT_DIR / 'overload_intervals.csv')
print('Saved:', OUTPUT_DIR / 'overload_masked_candidates.csv')
print('Saved:', OUTPUT_DIR / 'candidates_after_overload_mask.csv')
print('Saved:', OUTPUT_DIR / 'overload_summary.json')
print('Saved:', OUTPUT_DIR / 'overload_interval_overview.png')

## 10. 可选审计：用稀疏峰值轨迹识别固定窄带线

LoRa upchirp 会扫过约 41.667 kHz 占用带宽；持续载波则常反复落在同一两个粗 FFT bin。本节用“至少4个不同粗峰值bin、频率跨度至少25 kHz”建立一个可选的粗形态审计表，用来评估未来工程化时能否低成本缩小计算规模。


In [ ]:
candidates_before_sparse_morphology = [dict(row) for row in candidates]
morphology_retained, morphology_rejected = search.classify_sparse_morphology(
    candidates_before_sparse_morphology, config
)

pd.DataFrame(morphology_rejected).to_csv(
    OUTPUT_DIR / 'fixed_tone_like_rejected_candidates.csv', index=False
)
pd.DataFrame(morphology_retained).to_csv(
    OUTPUT_DIR / 'candidates_after_sparse_morphology.csv', index=False
)

known_candidate_ids = {
    candidate_id
    for check in known_checks
    for candidate_id in check.get('matching_candidate_ids', [])
}
input_known_ids = known_candidate_ids & {
    row['candidate_id'] for row in candidates_before_sparse_morphology
}
rejected_known_ids = input_known_ids & {
    row['candidate_id'] for row in morphology_rejected
}
if rejected_known_ids:
    print(
        'WARNING: known events missed by the optional sparse audit:',
        sorted(rejected_known_ids),
    )

sparse_morphology_priority_candidates = sorted(
    morphology_retained, key=lambda row: -float(row['peak_excess_db'])
)
# 正式流程不采用本节硬过滤：第11节继续接收全部非过载候选。
candidates = candidates_before_sparse_morphology
candidate_df = pd.DataFrame(candidates)
morphology_summary = {
    'optional_audit_input_count': len(candidates_before_sparse_morphology),
    'optional_fixed_tone_like_count': len(morphology_rejected),
    'optional_priority_candidate_count': len(
        sparse_morphology_priority_candidates
    ),
    'formal_chirp_input_count': len(candidates),
    'minimum_unique_coarse_peak_bins': config.morphology_minimum_unique_peak_bins,
    'minimum_coarse_peak_span_hz': config.morphology_minimum_peak_span_hz,
    'interpretation': (
        'Optional acceleration audit only; all non-overload candidates continue '
        'to the formal raw-STFT chirp search.'
    ),
}
search._write_json(OUTPUT_DIR / 'sparse_morphology_summary.json', morphology_summary)
display(pd.DataFrame([morphology_summary]))
display(pd.DataFrame(morphology_rejected).sort_values(
    'peak_excess_db', ascending=False
).head(30) if morphology_rejected else pd.DataFrame())
print('Saved:', OUTPUT_DIR / 'fixed_tone_like_rejected_candidates.csv')
print('Saved:', OUTPUT_DIR / 'candidates_after_sparse_morphology.csv')
print('Formal Section-11 input remains:', len(candidates))

## 11. 原始电压短窗的 LoRa chirp 斜线形态预筛

本节直接接收第9节留下的全部非过载候选；第10节只提供可选审计结果，不缩减这里的正式输入。目的是用很少的原始样本确认候选中是否真的存在斜chirp，避免对大量假候选做1.6 s DDC。

每个候选最多选择4个代表性稀疏命中（最早命中、最强命中及其他强命中），每个只读取20 ms原始电压。对短窗使用 `NFFT=131072`、`hop=65536` 的Hann窗STFT：频率bin间距约3.662 kHz，相邻谱帧间隔约0.1365 ms。只保留搜索中心±60 kHz，并用每频率中位数和每时刻中位数去除固定频响及宽带共模变化。

Starlink LoRa的期望upchirp斜率约为 `+6.78 kHz/ms`。程序在±30%范围内搜索3 ms直线轨迹，并同时计算水平线和反向斜线对照。当前预筛要求：

- 最佳正斜线平均强度 ≥ 1.5倍稳健局部标准差；
- 轨迹上至少40%像素高于2倍稳健局部标准差；
- 正斜线得分至少比水平/反向对照高0.6倍稳健局部标准差。

最新全量结果中，1193个非过载候选全部完成计算、错误数为0，只有 `C00731` 和 `C00778` 通过。Event 1三项指标约为 `(4.60, 1.00, 2.23)`，较弱的Event 2约为 `(2.05, 0.55, 1.19)`；1191个阴性候选各项最大值分别为 `(1.175, 0.227, 0.479)`，均未越过三项门限。即使把门限放宽为 `(1.0, 0.20, 0.30)`，仍只有这两个事件通过，说明当前样本存在清楚的经验分离；但这些阈值仍不是经注入-恢复实验确定的普适检出界限。

In [ ]:
# 11.1 初始化可中断的短窗chirp形态检测
current_chirp_morphology_signature = (
    tuple(sorted(row['candidate_id'] for row in candidates)),
    config.chirp_probe_window_seconds,
    config.chirp_probe_max_windows,
    config.chirp_stft_nfft,
    config.chirp_stft_hop,
    config.chirp_stft_half_width_hz,
    config.chirp_track_duration_seconds,
    config.chirp_expected_slope_hz_per_second,
    config.chirp_slope_tolerance_fraction,
    config.chirp_minimum_track_score_z,
    config.chirp_minimum_track_occupancy,
    config.chirp_minimum_control_margin_z,
)
if globals().get('chirp_morphology_signature') != current_chirp_morphology_signature:
    chirp_morphology_by_candidate = {}
    chirp_morphology_signature = current_chirp_morphology_signature
    print('Chirp-morphology candidate set or parameters changed: reset the cache.')
elif 'chirp_morphology_by_candidate' not in globals():
    chirp_morphology_by_candidate = {}

candidates_before_chirp_morphology = [dict(row) for row in candidates]
CHIRP_DIAGNOSTIC_DIR = OUTPUT_DIR / 'chirp_morphology_diagnostics'
print(
    f'Completed candidates already in memory: '
    f'{len(chirp_morphology_by_candidate)}/{len(candidates)}'
)

In [ ]:
# 11.2 执行或继续短窗原始STFT斜线搜索
for candidate_index, candidate in enumerate(
    candidates_before_chirp_morphology, start=1
):
    candidate_id = candidate['candidate_id']
    if candidate_id in chirp_morphology_by_candidate:
        continue
    print(
        f'[chirp {candidate_index}/{len(candidates_before_chirp_morphology)}] '
        f"{candidate_id} {candidate['dataset']} "
        f"{candidate['rf_frequency_hz'] / 1e6:.4f} MHz",
        flush=True,
    )
    result = search.probe_candidate_chirp_morphology(
        candidate,
        inventory,
        config,
        CHIRP_DIAGNOSTIC_DIR / f'{candidate_id}.png',
    )
    chirp_morphology_by_candidate[candidate_id] = result
    print(
        '  ->', result['classification'],
        'score =', result['best_track_score_z'],
        'occupancy =', result['best_track_occupancy'],
        'margin =', result['control_margin_z'],
    )

print(
    f'Completed chirp probes: '
    f'{len(chirp_morphology_by_candidate)}/'
    f'{len(candidates_before_chirp_morphology)}'
)

In [ ]:
# 11.3 汇总形态结果，只将斜线阳性候选送入DDC
if len(chirp_morphology_by_candidate) != len(candidates_before_chirp_morphology):
    raise RuntimeError('Finish all Section-11 chirp probes before filtering.')

chirp_morphology_rows = [
    chirp_morphology_by_candidate[row['candidate_id']]
    for row in candidates_before_chirp_morphology
]
chirp_positive_ids = {
    row['candidate_id'] for row in chirp_morphology_rows
    if row['classification'] == 'chirp_morphology_match'
}
chirp_rejected_candidates = [
    {**candidate, **chirp_morphology_by_candidate[candidate['candidate_id']]}
    for candidate in candidates_before_chirp_morphology
    if candidate['candidate_id'] not in chirp_positive_ids
]
candidates = [
    candidate for candidate in candidates_before_chirp_morphology
    if candidate['candidate_id'] in chirp_positive_ids
]
candidate_df = pd.DataFrame(candidates)

search._write_csv(
    OUTPUT_DIR / 'chirp_morphology_results.csv',
    chirp_morphology_rows, search.CHIRP_MORPHOLOGY_FIELDS,
)
pd.DataFrame(chirp_rejected_candidates).to_csv(
    OUTPUT_DIR / 'chirp_morphology_rejected_candidates.csv', index=False
)
pd.DataFrame(candidates).to_csv(
    OUTPUT_DIR / 'candidates_after_chirp_morphology.csv', index=False
)

input_known_ids = known_candidate_ids & {
    row['candidate_id'] for row in candidates_before_chirp_morphology
}
missed_known_ids = input_known_ids - chirp_positive_ids
if missed_known_ids:
    raise RuntimeError(
        f'Known-event chirp-morphology regression failed: {sorted(missed_known_ids)}'
    )

chirp_morphology_summary = {
    'candidate_count_before_chirp_morphology': len(
        candidates_before_chirp_morphology
    ),
    'chirp_morphology_match_count': len(chirp_positive_ids),
    'rejected_non_chirp_count': sum(
        row['classification'] == 'rejected_non_chirp_morphology'
        for row in chirp_morphology_rows
    ),
    'error_count': sum(
        row['classification'] == 'error' for row in chirp_morphology_rows
    ),
    'minimum_track_score_z': config.chirp_minimum_track_score_z,
    'minimum_track_occupancy': config.chirp_minimum_track_occupancy,
    'minimum_control_margin_z': config.chirp_minimum_control_margin_z,
    'full_ddc_candidate_count': len(candidates),
    'interpretation': (
        'Compute-saving morphology prefilter calibrated on two known events; '
        'not an injection-tested completeness boundary.'
    ),
}
search._write_json(
    OUTPUT_DIR / 'chirp_morphology_summary.json',
    chirp_morphology_summary,
)

chirp_display_df = pd.DataFrame(chirp_morphology_rows)
if not chirp_display_df.empty:
    chirp_display_df = chirp_display_df.sort_values(
        ['chirp_morphology_pass', 'best_track_score_z'],
        ascending=[False, False],
    )
display(chirp_display_df.head(50))
print(json.dumps(chirp_morphology_summary, ensure_ascii=False, indent=2))
print('Saved:', OUTPUT_DIR / 'chirp_morphology_results.csv')
print('Saved:', OUTPUT_DIR / 'candidates_after_chirp_morphology.csv')
print('Diagnostics:', CHIRP_DIAGNOSTIC_DIR)

## 12. 短窗 DDC 后的 LoRa 前导、SFD 与 Starlink sync word 预验证

本节只对第11节保留的斜线形态阳性候选读取短原始电压窗口；最新全量运行中只有 `C00731` 和 `C00778` 进入本节。窗口覆盖稀疏命中范围，两端各保留0.35 s，总长不少于1.6 s。它调用LFdemod现有DDC和LoRa重采样，然后使用已固定版本 `lora-phy==0.3.0` 依次要求：

1. 检出 15 个连续前导 upchirp；
2. 通过 downchirp SFD 完成包边界同步；
3. 两个 sync symbols 与 Starlink 配置 `0x12` 对应的 `(8, 16)` 一致。

本节到此停止，不解 PHY header、payload 或 CRC。只有 `starlink_sync_match` 进入后面 5.5 s 完整 LFdemod 验证；`error` 只表示未完成计算，不是物理非检出。这一格可中断，同一内核重跑会跳过已完成候选。

In [ ]:
current_preamble_probe_ids = tuple(
    sorted(row['candidate_id'] for row in candidates)
)
if globals().get('preamble_probe_candidate_ids') != current_preamble_probe_ids:
    preamble_probe_by_candidate = {}
    preamble_probe_candidate_ids = current_preamble_probe_ids
    print('Preamble-probe candidate set changed: reset the probe cache.')
elif 'preamble_probe_by_candidate' not in globals():
    preamble_probe_by_candidate = {}

for candidate_index, candidate in enumerate(candidates, start=1):
    candidate_id = candidate['candidate_id']
    if candidate_id in preamble_probe_by_candidate:
        continue
    print(
        f'[preamble {candidate_index}/{len(candidates)}] {candidate_id} '
        f"{candidate['dataset']} "
        f"{candidate['rf_frequency_hz'] / 1e6:.4f} MHz",
        flush=True,
    )
    result = search.probe_candidate_preamble(candidate, inventory, config)
    preamble_probe_by_candidate[candidate_id] = result
    print(
        '  ->', result['classification'],
        'preambles =', result['preamble_detected_count'],
        'sync matches =', result['starlink_sync_match_count'],
    )

print(
    f'Completed preamble probes: '
    f'{len(preamble_probe_by_candidate)}/{len(candidates)}'
)

In [ ]:
if len(preamble_probe_by_candidate) != len(candidates):
    raise RuntimeError('Finish all Section-12 preamble probes before filtering.')

candidates_before_preamble_probe = [dict(row) for row in candidates]
preamble_probe_rows = [
    preamble_probe_by_candidate[row['candidate_id']]
    for row in candidates_before_preamble_probe
]
positive_ids = {
    row['candidate_id'] for row in preamble_probe_rows
    if row['classification'] == 'starlink_sync_match'
}
rejected_preamble_candidates = [
    {**candidate, **preamble_probe_by_candidate[candidate['candidate_id']]}
    for candidate in candidates_before_preamble_probe
    if candidate['candidate_id'] not in positive_ids
]
candidates = [
    candidate for candidate in candidates_before_preamble_probe
    if candidate['candidate_id'] in positive_ids
]
candidate_df = pd.DataFrame(candidates)

search._write_csv(
    OUTPUT_DIR / 'preamble_probe_results.csv',
    preamble_probe_rows, search.PREAMBLE_PROBE_FIELDS,
)
pd.DataFrame(rejected_preamble_candidates).to_csv(
    OUTPUT_DIR / 'preamble_probe_rejected_candidates.csv', index=False
)
pd.DataFrame(candidates).to_csv(
    OUTPUT_DIR / 'candidates_after_preamble_probe.csv', index=False
)

input_known_ids = known_candidate_ids & {
    row['candidate_id'] for row in candidates_before_preamble_probe
}
missed_known_ids = input_known_ids - positive_ids
if missed_known_ids:
    raise RuntimeError(
        f'Known-event preamble regression failed: {sorted(missed_known_ids)}'
    )

preamble_summary = {
    'probe_count': len(preamble_probe_rows),
    'starlink_sync_match_count': sum(
        row['classification'] == 'starlink_sync_match'
        for row in preamble_probe_rows
    ),
    'lora_sync_other_count': sum(
        row['classification'] == 'lora_sync_other'
        for row in preamble_probe_rows
    ),
    'preamble_without_sfd_count': sum(
        row['classification'] == 'preamble_without_sfd'
        for row in preamble_probe_rows
    ),
    'no_preamble_count': sum(
        row['classification'] == 'no_preamble'
        for row in preamble_probe_rows
    ),
    'error_count': sum(
        row['classification'] == 'error' for row in preamble_probe_rows
    ),
    'full_decode_candidate_count': len(candidates),
}
search._write_json(OUTPUT_DIR / 'preamble_probe_summary.json', preamble_summary)
display(pd.DataFrame(preamble_probe_rows))
print(json.dumps(preamble_summary, ensure_ascii=False, indent=2))
print('Saved:', OUTPUT_DIR / 'preamble_probe_results.csv')
print('Saved:', OUTPUT_DIR / 'candidates_after_preamble_probe.csv')

## 13. 初始化可中断 LFdemod 完整验证状态

每个候选使用默认 5.5 s 原始电压窗口，而不是 0.1365 ms 的搜索快照。若窗口跨越连续文件边界，程序会在系统临时目录拼接所需原始字节；临时文件验证后自动删除。

验证状态按候选 ID 保存在内存。中断后重新运行下一格，会跳过本内核会话中已完成的候选。若希望彻底重做，请更换 `RUN_NAME`，以免与已经保留的详细事件目录冲突。

In [ ]:
current_full_verification_ids = tuple(
    sorted(row['candidate_id'] for row in candidates)
)
if globals().get('full_verification_candidate_ids') != current_full_verification_ids:
    verification_by_candidate = {}
    packets_by_candidate = {}
    full_verification_candidate_ids = current_full_verification_ids
    print('Full-verification candidate set changed: reset the cache.')
elif 'verification_by_candidate' not in globals():
    verification_by_candidate = {}
    packets_by_candidate = {}

# 若要在当前内核中重置验证状态，请更换 RUN_NAME 后再执行下面两行。
# verification_by_candidate = {}
# packets_by_candidate = {}

print(f'Completed candidates already in memory: {len(verification_by_candidate)}/{len(candidates)}')

## 14. 执行或继续 LFdemod 完整验证

LFdemod 负责完整的 DDC、LoRa 同步与解调、显式 PHY header、payload CRC 和 Starlink profile 检查。Notebook 不复制这些底层算法。

分类：

- `strict`：至少一个 packet 通过 Starlink VHF profile；
- `likely`：解出 LoRa packet，但没有通过全部 Starlink profile；
- `rejected_energy_only`：只有能量候选，没有解出 packet；
- `error`：处理错误，不能解释为物理非检出。

只有 `strict` 和 `likely` 保留完整 LFdemod 文件；阴性能量候选最终只保留汇总行。

In [ ]:
for candidate_index, candidate in enumerate(candidates, start=1):
    candidate_id = candidate['candidate_id']
    if candidate_id in verification_by_candidate:
        continue
    print(
        f"[{candidate_index}/{len(candidates)}] {candidate_id} "
        f"{candidate['dataset']}  "
        f"{candidate['rf_frequency_hz'] / 1e6:.4f} MHz",
        flush=True,
    )
    result, packet_rows = search.verify_one_candidate(
        candidate,
        inventory,
        OUTPUT_DIR,
        config,
        RETAIN_IQ,
    )
    verification_by_candidate[candidate_id] = result
    packets_by_candidate[candidate_id] = packet_rows
    print(
        '  ->', result['classification'],
        'packets =', result['packet_count'],
        'profile matches =', result['starlink_profile_match_count'],
    )

print(f'Completed candidates: {len(verification_by_candidate)}/{len(candidates)}')

## 15. 汇总、物理事件去重并保存最终结果

137.8463 与 137.8763 MHz 只相隔 30 kHz，小于当前约 41.667 kHz LoRa 占用带宽，因此同一物理包可能在两个调谐中心被重复解出。程序结合数据集、preamble 时间、payload 和观测中心频率去重，严格 packet 优先于可能 packet。

In [ ]:
if len(verification_by_candidate) != len(candidates):
    raise RuntimeError(
        f'Only {len(verification_by_candidate)}/{len(candidates)} candidates '
        'have been verified. Finish Section 14 before finalizing.'
    )

verification_rows = [
    verification_by_candidate[candidate['candidate_id']]
    for candidate in candidates
]
packet_rows = [
    row
    for candidate in candidates
    for row in packets_by_candidate[candidate['candidate_id']]
]
unique_packets = search.deduplicate_packet_rows(
    packet_rows,
    config.deduplication_time_tolerance_seconds,
)
strict_packets = [row for row in unique_packets if row['classification'] == 'strict']
likely_packets = [row for row in unique_packets if row['classification'] == 'likely']

search._write_csv(
    OUTPUT_DIR / 'verification_results.csv',
    verification_rows,
    search.VERIFICATION_FIELDS,
)
search._write_csv(OUTPUT_DIR / 'detections.csv', strict_packets, search.PACKET_FIELDS)
search._write_csv(OUTPUT_DIR / 'likely_events.csv', likely_packets, search.PACKET_FIELDS)
search.save_verification_overview(
    OUTPUT_DIR / 'verification_overview.png',
    verification_rows,
    DATASETS,
    config.channels_hz,
)

classification_counts = pd.Series(
    [row['classification'] for row in verification_rows]
).value_counts().to_dict() if verification_rows else {}
final_summary = {
    'candidate_count': len(candidates),
    'candidate_classification_counts': classification_counts,
    'deduplicated_strict_packet_count': len(strict_packets),
    'deduplicated_likely_packet_count': len(likely_packets),
    'retained_detailed_output_policy': (
        'Detailed LFdemod artifacts retained only for strict and likely classifications.'
    ),
    'scope_limit': (
        'No injection/recovery completeness measurement was performed. '
        'A threshold-dependent non-detection is not an absolute proof of no transmission.'
    ),
}
search._write_json(OUTPUT_DIR / 'search_summary.json', final_summary)

display(pd.DataFrame(verification_rows))
print(json.dumps(final_summary, ensure_ascii=False, indent=2))
display(Image(filename=str(OUTPUT_DIR / 'verification_overview.png')))

## 16. 查看严格检出与可能事件

`detections.csv` 是去重后的严格 Starlink-profile packet；`likely_events.csv` 是解出 LoRa PHY 但没有通过全部 profile 的 packet。若表为空，表示在本次参数和阈值下没有相应分类，并不自动等于物理上的绝对无发射。

In [ ]:
print('Strict detections')
display(pd.DataFrame(strict_packets))

print('Likely LoRa events')
display(pd.DataFrame(likely_packets))

print('Detailed retained directories:')
for path in sorted((OUTPUT_DIR / 'detailed').glob('*/*')) if (OUTPUT_DIR / 'detailed').exists() else []:
    print(' ', path)